# 04 — Pilot VLM: Direct (D) vs Structured reasoning (E)

Notebook này chạy thử nghiệm sơ bộ cho phương pháp đề xuất trên một mẫu ngẫu nhiên của tập **val** EMOTIC (PAMI):

- **D — Direct VLM**: hỏi thẳng VLM người trong khung đỏ có những cảm xúc nào trong 26 nhãn.
- **E — Structured VLM**: bắt VLM trả JSON theo thứ tự cố định: dấu hiệu từ người → dấu hiệu ngữ cảnh → tình huống → cảm xúc.

Cả hai dùng cùng model, cùng ảnh, cùng khung đỏ đánh dấu người mục tiêu, cùng giới hạn token, giải mã greedy. Khác nhau ở prompt: E thêm yêu cầu nêu bằng chứng nhìn thấy, mô tả tình huống và trả theo schema JSON. Vì vậy chênh lệch đo được là chênh lệch giữa hai cấu hình prompt, chưa tách riêng tác động của từng thành phần trong E.

Kết quả ghi từng mẫu vào JSONL trên Drive, nên runtime bị ngắt thì chạy lại với `RUN_ID` cũ để chạy tiếp.

## 1. Mount Drive, clone mã nguồn và cài dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Contextual Emotion Recognition"
)
REPO_URL = (
    "https://github.com/minh-1105/"
    "Explainable-Contextual-Emotion-Recognition-via-"
    "Text-Guided-Chain-of-Thought-Reasoning.git"
)
CODE_ROOT = Path("/content/contextual-emotion-reasoning")

assert DRIVE_ROOT.is_dir(), f"Không tìm thấy project trên Drive: {DRIVE_ROOT}"

if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements.txt")],
    check=True,
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

import transformers
print("transformers:", transformers.__version__)

## 2. Cấu hình

`N_SAMPLES` người được lấy ngẫu nhiên (seed cố định) từ tập val. Để chạy tiếp một run bị ngắt, đặt `RUN_ID` thành tên thư mục run cũ.

Model mặc định `Qwen/Qwen2.5-VL-3B-Instruct` vừa GPU T4 (fp16). Nếu có L4/A100, có thể đổi sang `Qwen/Qwen2.5-VL-7B-Instruct`.

In [ ]:
from datetime import datetime
import json
import shutil
import time
import zipfile

EMOTIC_ROOT = DRIVE_ROOT / "data/raw/raw/emotic"
PAMI_ROOT = EMOTIC_ROOT / "PAMI"
CVPR_ROOT = EMOTIC_ROOT / "CVPR"
ANNOTATION_PATH = PAMI_ROOT / "Annotations/Annotations/Annotations.mat"

# Ảnh lưu trên Drive dưới dạng zip và được giải nén vào /content, vì đọc
# hàng chục nghìn ảnh nhỏ trực tiếp từ Drive rất chậm.
LOCAL_EMOTIC_ROOT = Path("/content/emotic")
IMAGE_ARCHIVES = (
    # (zip trên Drive, thư mục giải nén, thư mục gốc trong zip, số file ảnh)
    (PAMI_ROOT / "pami_emotic.zip", LOCAL_EMOTIC_ROOT / "PAMI", "emotic", 23185),
    (CVPR_ROOT / "cvpr_emotic.zip", LOCAL_EMOTIC_ROOT / "CVPR", "cvpr_emotic", 18302),
)

PILOT_CONFIG = {
    "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
    "split": "val",
    "n_samples": 300,
    "seed": 42,
    "category_policy": "union",
    "max_pixels": 640 * 640,
    "max_new_tokens": 384,
    "modes": ["direct", "structured"],
}

# Ví dụ: RUN_ID = "20261019-200000"
RUN_ID = None


def extract_images(archive_path, extract_dir, root_name, expected_files):
    """Giải nén zip ảnh từ Drive vào ổ local của Colab, mỗi runtime một lần."""
    image_root = extract_dir / root_name
    marker = extract_dir / ".extracted"
    if marker.is_file():
        return image_root
    assert archive_path.is_file(), f"Thiếu file zip: {archive_path}"
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True)
    started = time.time()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
    file_count = sum(1 for path in image_root.rglob("*") if path.is_file())
    assert file_count == expected_files, (
        f"{archive_path.name}: {file_count} file, cần {expected_files}"
    )
    marker.touch()
    print(f"Giải nén {archive_path.name}: {file_count:,} file, {time.time() - started:.0f}s")
    return image_root


assert ANNOTATION_PATH.is_file(), f"Thiếu annotation: {ANNOTATION_PATH}"
IMAGE_ROOTS = tuple(extract_images(*spec) for spec in IMAGE_ARCHIVES)

PILOT_ROOT = DRIVE_ROOT / "outputs/vlm_pilot"
if RUN_ID is None:
    RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
    RUN_DIR = PILOT_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=True)
    (RUN_DIR / "config.json").write_text(
        json.dumps(PILOT_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8"
    )
else:
    RUN_DIR = PILOT_ROOT / RUN_ID
    saved_config = json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8"))
    assert saved_config == PILOT_CONFIG, "Cấu hình khác run cũ; tạo run mới thay vì resume."

print("Run dir:", RUN_DIR)

## 3. Lấy mẫu người từ tập val

Mỗi người là một mẫu (ảnh nhiều người cho nhiều mẫu). Danh sách `sample_id` được ghi ra `samples.json` để D và E dùng đúng cùng một tập.

In [ ]:
import random
from src.datasets.emotic_dataset import EMOTION_LABELS, EmoticDataset

val_dataset = EmoticDataset(
    annotation_path=ANNOTATION_PATH,
    split=PILOT_CONFIG["split"],
    image_roots=IMAGE_ROOTS,
    category_policy=PILOT_CONFIG["category_policy"],
    strict_images=True,
)
sample_indices = sorted(
    random.Random(PILOT_CONFIG["seed"]).sample(range(len(val_dataset)), PILOT_CONFIG["n_samples"])
)
sample_ids = [val_dataset.samples[i].sample_id for i in sample_indices]

samples_path = RUN_DIR / "samples.json"
if samples_path.is_file():
    assert json.loads(samples_path.read_text(encoding="utf-8")) == sample_ids
else:
    samples_path.write_text(json.dumps(sample_ids, indent=2), encoding="utf-8")

print(f"Val: {len(val_dataset):,} người; pilot: {len(sample_indices)} người")

## 4. Prompt D và E

Người mục tiêu được đánh dấu bằng khung đỏ vẽ lên ảnh toàn cảnh, vì VLM chỉ nhận ảnh và văn bản. Hai prompt dùng chung danh sách nhãn và cùng yêu cầu trả JSON; E bắt buộc nêu bằng chứng nhìn thấy được trước khi kết luận.

In [ ]:
import re
from PIL import ImageDraw

LABEL_LIST = ", ".join(EMOTION_LABELS)
TARGET = "The target person is the one inside the red box."

PROMPTS = {
    "direct": (
        f"{TARGET} Which emotions does the target person feel? "
        f"Choose every category that applies from this list: {LABEL_LIST}.\n"
        'Answer with JSON only, in the form {"emotions": ["...", "..."]}.'
    ),
    "structured": (
        f"{TARGET} Infer the emotions of the target person step by step. "
        "First describe what is visible, then conclude.\n"
        "Answer with JSON only, using exactly these keys in this order:\n"
        '{"person_cues": ["visible cues from the target person: face, body pose, gestures, action"],\n'
        ' "context_cues": ["visible cues from the scene, objects and other people"],\n'
        ' "situation": "one sentence on what is happening to the target person",\n'
        ' "emotions": ["categories from the list"]}\n'
        f"Choose every emotion category that applies from this list: {LABEL_LIST}.\n"
        "Only mention cues that are actually visible in the image."
    ),
}


def mark_target(image, bbox):
    marked = image.copy()
    width = max(3, round(max(marked.size) / 150))
    ImageDraw.Draw(marked).rectangle([float(v) for v in bbox], outline=(255, 0, 0), width=width)
    return marked


_LABEL_KEYS = {re.sub(r"[^a-z]", "", label.lower()): label for label in EMOTION_LABELS}
_LABEL_KEYS.update({"doubt": "Doubt/Confusion", "confusion": "Doubt/Confusion"})
SCHEMA_KEYS = {"direct": ["emotions"], "structured": ["person_cues", "context_cues", "situation", "emotions"]}


def parse_response(text):
    """Đọc object JSON đầu tiên trong câu trả lời.

    Trả (dict JSON hoặc None, danh sách nhãn hợp lệ, danh sách nhãn lạ).
    """
    start = text.find("{")
    if start < 0:
        return None, [], []
    try:
        payload, _ = json.JSONDecoder().raw_decode(text[start:])
    except json.JSONDecodeError:
        return None, [], []
    emotions = payload.get("emotions") if isinstance(payload, dict) else None
    if not isinstance(emotions, list):
        return None, [], []
    labels, unknown = [], []
    for item in emotions:
        key = re.sub(r"[^a-z]", "", str(item).lower())
        if key in _LABEL_KEYS:
            if _LABEL_KEYS[key] not in labels:
                labels.append(_LABEL_KEYS[key])
        else:
            unknown.append(str(item))
    return payload, labels, unknown


def _text_list(value):
    return isinstance(value, list) and len(value) > 0 and all(isinstance(v, str) and v.strip() for v in value)


def schema_ok(mode, payload):
    """Đúng tập khóa theo thứ tự của schema, đúng kiểu, bằng chứng và tình huống không rỗng."""
    if not isinstance(payload, dict) or list(payload) != SCHEMA_KEYS[mode]:
        return False
    if not isinstance(payload["emotions"], list):
        return False
    if mode == "direct":
        return True
    situation = payload["situation"]
    return (_text_list(payload["person_cues"]) and _text_list(payload["context_cues"])
            and isinstance(situation, str) and bool(situation.strip()))


print(PROMPTS["structured"])

## 5. Nạp model

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "Hãy bật GPU: Runtime > Change runtime type > GPU"
capability = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if capability[0] >= 8 else torch.float16

processor = AutoProcessor.from_pretrained(
    PILOT_CONFIG["model_id"], max_pixels=PILOT_CONFIG["max_pixels"]
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    PILOT_CONFIG["model_id"], torch_dtype=dtype, attn_implementation="sdpa"
).to("cuda").eval()

commit = subprocess.run(
    ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
(RUN_DIR / "environment.txt").write_text(
    f"transformers={transformers.__version__}\ntorch={torch.__version__}\n"
    f"gpu={torch.cuda.get_device_name(0)}\ndtype={dtype}\ngit_commit={commit}\n",
    encoding="utf-8",
)
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", dtype)


@torch.inference_mode()
def ask_vlm(image, prompt, max_new_tokens):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    return text, generated.shape[1] >= max_new_tokens

## 6. Smoke test: 2 mẫu cho mỗi prompt

Đọc output thô bên dưới trước khi chạy toàn bộ. Nếu model không trả JSON đúng dạng, dừng lại sửa prompt thay vì chạy tiếp.

In [ ]:
for index in sample_indices[:2]:
    item = val_dataset[index]
    image = mark_target(item["context_image"], item["bbox"].tolist())
    truth = [EMOTION_LABELS[i] for i, v in enumerate(item["labels"].tolist()) if v > 0]
    print("=" * 80, "\nGround truth:", truth)
    for mode in PILOT_CONFIG["modes"]:
        raw, truncated = ask_vlm(image, PROMPTS[mode], PILOT_CONFIG["max_new_tokens"])
        payload, labels, unknown = parse_response(raw)
        print(f"--- {mode} | schema={'OK' if schema_ok(mode, payload) else 'FAIL'} | truncated={truncated} "
              f"| labels={labels} | unknown={unknown}")
        print(raw)

## 7. Chạy pilot

Mỗi mẫu được ghi ngay vào `predictions_<mode>.jsonl`. Chạy lại cell này sẽ bỏ qua các mẫu đã có.

In [ ]:
from tqdm.auto import tqdm

for mode in PILOT_CONFIG["modes"]:
    path = RUN_DIR / f"predictions_{mode}.jsonl"
    done = set()
    if path.is_file():
        done = {json.loads(line)["sample_id"] for line in path.read_text(encoding="utf-8").splitlines()}
    todo = [i for i in sample_indices if val_dataset.samples[i].sample_id not in done]
    print(f"{mode}: {len(done)} đã có, còn {len(todo)}")
    with path.open("a", encoding="utf-8") as file:
        for index in tqdm(todo, desc=mode):
            item = val_dataset[index]
            image = mark_target(item["context_image"], item["bbox"].tolist())
            started = time.time()
            raw, truncated = ask_vlm(image, PROMPTS[mode], PILOT_CONFIG["max_new_tokens"])
            payload, labels, unknown = parse_response(raw)
            record = {
                "sample_id": item["sample_id"],
                "relative_image_path": item["relative_image_path"],
                "bbox": item["bbox"].tolist(),
                "truth": [EMOTION_LABELS[i] for i, v in enumerate(item["labels"].tolist()) if v > 0],
                "predicted": labels,
                "unknown_labels": unknown,
                "parse_ok": payload is not None,
                "schema_ok": schema_ok(mode, payload),
                "truncated": truncated,
                "response": payload,
                "raw": raw,
                "seconds": round(time.time() - started, 2),
            }
            file.write(json.dumps(record, ensure_ascii=False) + "\n")
            file.flush()

## 8. Metrics

- Mẫu không parse được tính là dự đoán rỗng (không bỏ khỏi mẫu số).
- Macro F1 lấy trung bình trên các nhãn có ít nhất một người mang nhãn đó trong mẫu; nhãn vắng mặt bị loại thay vì tính F1 = 0.
- `schema_rate`: câu trả lời có đúng tập khóa theo thứ tự của schema, đúng kiểu, bằng chứng và tình huống không rỗng. Chỉ số này đo hình thức, không đo bằng chứng có đúng hay không.
- Khoảng tin cậy 95% của chênh lệch macro F1 (E − D): paired bootstrap 2.000 lần, lấy mẫu lại theo **ảnh** (giữ nguyên mọi người trong cùng ảnh), vì những người chung ảnh không độc lập. CI chỉ phản ánh biến thiên do lấy mẫu người/ảnh, không phản ánh biến thiên giữa model hay cách viết prompt.

In [ ]:
import numpy as np
from collections import defaultdict
from sklearn.metrics import f1_score


def load_records(mode):
    path = RUN_DIR / f"predictions_{mode}.jsonl"
    records = {r["sample_id"]: r for r in map(json.loads, path.read_text(encoding="utf-8").splitlines())}
    assert set(records) == set(sample_ids), f"{mode}: thiếu mẫu, chạy lại cell 7"
    return [records[sample_id] for sample_id in sample_ids]


def to_multi_hot(label_lists):
    matrix = np.zeros((len(label_lists), len(EMOTION_LABELS)), dtype=np.int32)
    for row, labels in enumerate(label_lists):
        for label in labels:
            matrix[row, EMOTION_LABELS.index(label)] = 1
    return matrix


records = {mode: load_records(mode) for mode in PILOT_CONFIG["modes"]}
truth = to_multi_hot([r["truth"] for r in records["direct"]])
predictions = {mode: to_multi_hot([r["predicted"] for r in rs]) for mode, rs in records.items()}
present_labels = [i for i in range(len(EMOTION_LABELS)) if truth[:, i].sum() > 0]


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=present_labels, average="macro", zero_division=0)


summary = {"n_people": len(sample_ids), "n_images": len({r["relative_image_path"] for r in records["direct"]}),
           "n_labels_present": len(present_labels), "max_new_tokens": PILOT_CONFIG["max_new_tokens"]}
for mode, rs in records.items():
    pred = predictions[mode]
    summary[mode] = {
        "n": len(rs),
        "macro_f1": float(macro_f1(truth, pred)),
        "micro_f1": float(f1_score(truth, pred, average="micro", zero_division=0)),
        "samples_f1": float(f1_score(truth, pred, average="samples", zero_division=0)),
        "parse_rate": float(np.mean([r["parse_ok"] for r in rs])),
        "schema_rate": float(np.mean([r["schema_ok"] for r in rs])),
        "truncated_rate": float(np.mean([r["truncated"] for r in rs])),
        "mean_predicted_labels": float(pred.sum(1).mean()),
        "mean_true_labels": float(truth.sum(1).mean()),
        "unknown_label_rate": float(np.mean([bool(r["unknown_labels"]) for r in rs])),
        "mean_seconds": float(np.mean([r["seconds"] for r in rs])),
    }

people_by_image = defaultdict(list)
for row, record in enumerate(records["direct"]):
    people_by_image[record["relative_image_path"]].append(row)
image_groups = list(people_by_image.values())

rng = np.random.default_rng(PILOT_CONFIG["seed"])
deltas = []
for _ in range(2000):
    picked = rng.integers(0, len(image_groups), len(image_groups))
    idx = np.concatenate([image_groups[g] for g in picked])
    deltas.append(macro_f1(truth[idx], predictions["structured"][idx])
                  - macro_f1(truth[idx], predictions["direct"][idx]))
summary["delta_macro_f1_structured_minus_direct"] = {
    "point": summary["structured"]["macro_f1"] - summary["direct"]["macro_f1"],
    "ci95": [float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))],
    "bootstrap": "paired, resampling images, 2000 draws",
}

(RUN_DIR / "metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))